In [1]:
# Set up the file path
import os
os.chdir('..')

In [2]:
# Import packages
from RL4CRN.policies.parameter_generator_from_distribution import ParameterGeneratorFromDistribution
from RL4CRN.iocrns.reaction_library import construct_hill_production_library
import torch

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Create a library
library = construct_hill_production_library(species_labels=['X_1', 'X_2', 'X_3'], max_product_order=1, max_num_regulators=2)
M = len(library.reactions) # Number of possible reactions

In [ ]:
# Construct the continuous parameter masks
discrete_parameter_mask = library.get_parameter_mask(mode="discrete")
discrete_parameter_mask = torch.tensor(discrete_parameter_mask, dtype=torch.float32).to(device) if discrete_parameter_mask is not None else None # Shape: (M, max_num_discrete_parameters)
D = discrete_parameter_mask.shape[1] if discrete_parameter_mask is not None else 0 # Number of discrete parameters per reaction

TypeError: 'NoneType' object is not iterable

In [ ]:
print(library)
print(discrete_parameter_mask)

In [ ]:
# Create an instance of ParameterGeneratorFromDistribution
d = 1000
h = 128
n = 3
discrete_parameter_generator = ParameterGeneratorFromDistribution(
    distribution={"type": 'categorical', "dim": D, "categories": torch.tensor([1, 2, 3, 4])}, 
    backbone_attributes={"input_size": d + M, 
                         "hidden_size": h, 
                         "num_layers": n
                        }, 
    device=device
    ).to(device=device)

In [ ]:
# Generate a batch of reaction indices
N = 4
samples_reaction_idx = torch.randint(low=0, high=M, size=(N,), device=device)

# Map the masks to the sampled reaction indices
discrete_parameter_mask_subset = discrete_parameter_mask[samples_reaction_idx] if discrete_parameter_mask is not None else None

In [ ]:
# Construct a batch of input data to test the forward pass
x = torch.randn((N, d + M), device=device) 
samples, log_probs, entropies = discrete_parameter_generator(x, discrete_parameter_mask_subset) 
print(discrete_parameter_mask_subset)
print(samples)

In [ ]:
from RL4CRN.distributions.lognormal import MultivariateLogNormal
mask = torch.tensor([1.0, 1.0, 1.0, 0.0])
m = torch.tensor([1.0, 2.0, 3.0, 4.0])
S = 0.001*torch.eye(4)
outer_mm = m.unsqueeze(-1) * m.unsqueeze(-2)                            # shape: (N, D, D)
Sigma = torch.log1p(S / outer_mm)                                       # shape: (N, D, D)
mu = torch.log(m) - 0.5 * torch.diagonal(Sigma, dim1=-1, dim2=-2)

mu = mu.masked_fill(~mask.bool(), float('-inf'))
mask_soft = torch.where(mask == 0, torch.tensor(1e-8, dtype=mask.dtype, device=mask.device), mask)
Sigma = Sigma * mask_soft.unsqueeze(-1) * mask_soft.unsqueeze(-2)

dist = MultivariateLogNormal(loc=mu, covariance_matrix=Sigma)
samples = dist.sample((10,))

In [ ]:
print(samples)
print(mask)